# 01 · SQL: perguntas de negócio
[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jessicapmartins/segmentacao-clientes-olist/blob/main/notebooks/01_sql_perguntas_negocio.ipynb)

Cada pergunta vira **uma consulta SQL** e **uma frase de conclusão**. A pergunta 1 está resolvida como exemplo; as demais são suas.

**Dica de estudo:** escreva primeiro em português o que a consulta precisa fazer (de onde vem, como junta, como agrupa), depois o SQL.

Modelo das tabelas: `clientes` 1→N `pedidos` 1→N `itens_pedido` N→1 `produtos` · `pedidos` 1→N `pagamentos` · `pedidos` 1→N `avaliacoes` · `itens_pedido` N→1 `vendedores`.

In [1]:
import sqlite3, pandas as pd
from google.colab import drive
drive.mount('/content/drive')
DADOS = '/content/drive/MyDrive/03_Portfolio/Projetos_ML/01_segmentacao-clientes-olist/dados'
con = sqlite3.connect(f'{DADOS}/olist.db')
def q(sql):
    """Roda uma consulta SQL no banco da Olist e devolve um DataFrame."""
    return pd.read_sql(sql, con)

Mounted at /content/drive


In [9]:
# DICIONÁRIO DA BASE: cada tabela, suas colunas e um exemplo de valor
for tabela in ['clientes', 'pedidos', 'itens_pedido', 'pagamentos',
               'avaliacoes', 'produtos', 'vendedores', 'categorias_en']:
    total = q(f"SELECT COUNT(*) AS n FROM {tabela}")['n'][0]
    exemplo = q(f"SELECT * FROM {tabela} LIMIT 1").T      # T = vira a tabela de lado
    exemplo.columns = ['exemplo']
    print(f"\n===== {tabela.upper()}  ({total:,} linhas) =====".replace(',', '.'))
    display(exemplo)


===== CLIENTES  (99.441 linhas) =====


,exemplo
customer_id,06b8999e2fba1a1fbc88172c00ba8bc7
customer_unique_id,861eff4711a542e4b93843c6dd7febb0
customer_zip_code_prefix,14409
customer_city,franca
customer_state,SP



===== PEDIDOS  (99.441 linhas) =====


,exemplo
order_id,e481f51cbdc54678b7cc49136f2d6af7
customer_id,9ef432eb6251297304e76186b10a928d
order_status,delivered
order_purchase_timestamp,2017-10-02 10:56:33
order_approved_at,2017-10-02 11:07:15
order_delivered_carrier_date,2017-10-04 19:55:00
order_delivered_customer_date,2017-10-10 21:25:13
order_estimated_delivery_date,2017-10-18 00:00:00



===== ITENS_PEDIDO  (112.650 linhas) =====


...


===== VENDEDORES  (3.095 linhas) =====


,exemplo
seller_id,3442f8959a84dea7ee197c632cb2df15
seller_zip_code_prefix,13023
seller_city,campinas
seller_state,SP



===== CATEGORIAS_EN  (71 linhas) =====


,exemplo
product_category_name,beleza_saude
product_category_name_english,health_beauty


Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


## 1. Tamanho do negócio *(exemplo resolvido)*
Quantos pedidos, clientes únicos e vendedores existem?

In [2]:
q('''
SELECT
    (SELECT COUNT(DISTINCT order_id)           FROM pedidos)    AS pedidos,
    (SELECT COUNT(DISTINCT customer_unique_id) FROM clientes)   AS clientes_unicos,
    (SELECT COUNT(DISTINCT seller_id)          FROM vendedores) AS vendedores
''')

,pedidos,clientes_unicos,vendedores
0,99441,96096,3095


**Conclusão:** _escreva aqui em uma frase._

## 2. Faturamento mensal e crescimento
Receita (soma de `price` + `freight_value`) por mês da compra e variação % contra o mês anterior.

*Conceitos:* `strftime('%Y-%m', ...)`, CTE (`WITH`), função de janela `LAG()`.

In [6]:
q('''
-- ETAPA 1: tabela temporária com a receita de cada mês
WITH receita_mensal AS (
    SELECT
        strftime('%Y-%m', p.order_purchase_timestamp) AS mes,      -- transforma a data em "2017-05"
        SUM(i.price) + SUM(i.freight_value)          AS receita   -- sua lógica: produto + frete
    FROM pedidos p
    JOIN itens_pedido i ON p.order_id = i.order_id                -- liga cada pedido aos seus itens
    WHERE p.order_status = 'delivered'                            -- só pedidos entregues (ver nota abaixo)
    GROUP BY mes
)

-- ETAPA 2: usa a tabela da etapa 1 e compara cada mês com o anterior
SELECT
    mes,
    ROUND(receita, 2)                                   AS receita,
    ROUND(LAG(receita) OVER (ORDER BY mes), 2)          AS receita_mes_anterior,  -- valor da linha de cima
    ROUND(
        100.0 * (receita - LAG(receita) OVER (ORDER BY mes))
              / LAG(receita) OVER (ORDER BY mes)
    , 1)                                                AS variacao_pct           -- crescimento em %
FROM receita_mensal
ORDER BY mes
''')

,mes,receita,receita_mes_anterior,variacao_pct
0,2016-09,143.46,NaN,NaN
1,2016-10,46490.66,143.46,32306.7
2,2016-12,19.62,46490.66,-100.0
3,2017-01,127482.37,19.62,649657.2
4,2017-02,271239.32,127482.37,112.8
5,2017-03,414330.95,271239.32,52.8
6,2017-04,390812.40,414330.95,-5.7
7,2017-05,566851.40,390812.40,45.0
8,2017-06,490050.37,566851.40,-13.5
9,2017-07,566299.08,490050.37,15.6


## 3. Top 10 categorias por receita
Use a tabela `categorias_en` para o nome em inglês e `DENSE_RANK()` para o ranking.

*Conceitos:* JOIN de 3 tabelas, `DENSE_RANK() OVER (ORDER BY ...)`.

In [7]:
q('''
-- ETAPA 1: receita e nº de pedidos por categoria
WITH receita_categoria AS (
    SELECT
        COALESCE(c.product_category_name_english,      -- nome em inglês, se existir
                 pr.product_category_name,             -- senão, o nome em português
                 'sem_categoria')                      AS categoria,  -- senão, marca como sem categoria
        SUM(i.price) + SUM(i.freight_value)            AS receita,
        COUNT(DISTINCT i.order_id)                     AS pedidos
    FROM pedidos p
    JOIN itens_pedido i       ON p.order_id = i.order_id                    -- pedido → itens
    JOIN produtos pr          ON i.product_id = pr.product_id               -- item → produto
    LEFT JOIN categorias_en c ON pr.product_category_name = c.product_category_name  -- tradução
    WHERE p.order_status = 'delivered'                 -- mesmo critério da pergunta 2
    GROUP BY categoria
),

-- ETAPA 2: cria o ranking (posição 1 = maior receita)
ranking AS (
    SELECT
        categoria,
        ROUND(receita, 2)                                        AS receita,
        pedidos,
        ROUND(100.0 * receita / SUM(receita) OVER (), 1)         AS participacao_pct,  -- % da receita total
        DENSE_RANK() OVER (ORDER BY receita DESC)                AS posicao
    FROM receita_categoria
)

-- ETAPA 3: só as 10 primeiras
SELECT *
FROM ranking
WHERE posicao <= 10
ORDER BY posicao
''')

,categoria,receita,pedidos,participacao_pct,posicao
0,health_beauty,1412271.15,8647,9.2,1
1,watches_gifts,1264333.12,5495,8.2,2
2,bed_bath_table,1225209.26,9272,7.9,3
3,sports_leisure,1119923.63,7530,7.3,4
4,computers_accessories,1032723.77,6530,6.7,5
5,furniture_decor,881063.91,6307,5.7,6
6,housewares,759313.29,5743,4.9,7
7,cool_stuff,691680.89,3559,4.5,8
8,auto,669559.87,3810,4.3,9
9,garden_tools,567354.11,3448,3.7,10


## 4. Ticket médio por estado do cliente
*Conceitos:* JOIN `clientes` × `pedidos` × `pagamentos`, `GROUP BY`, `HAVING` para tirar estados com poucos pedidos.

In [8]:
q('''
SELECT
    c.customer_state                                        AS estado,
    COUNT(DISTINCT p.order_id)                              AS pedidos,
    ROUND(SUM(pg.payment_value), 2)                         AS receita,
    ROUND(SUM(pg.payment_value)
          / COUNT(DISTINCT p.order_id), 2)                  AS ticket_medio   -- valor total ÷ nº de pedidos
FROM clientes c
JOIN pedidos p     ON c.customer_id = p.customer_id         -- cliente → pedido
JOIN pagamentos pg ON p.order_id   = pg.order_id            -- pedido → pagamento(s)
WHERE p.order_status = 'delivered'                          -- filtra LINHAS (antes de agrupar)
GROUP BY c.customer_state
HAVING COUNT(DISTINCT p.order_id) >= 100                    -- filtra GRUPOS (depois de agrupar)
ORDER BY ticket_medio DESC
''')

,estado,pedidos,receita,ticket_medio
0,PB,517,137834.65,266.60
1,AL,397,94195.79,237.27
2,RO,243,56975.70,234.47
3,PA,946,212027.55,224.13
4,PI,476,105272.17,221.16
5,TO,274,60007.37,219.00
6,RN,474,100728.30,212.51
7,SE,335,70289.13,209.82
8,CE,1279,266463.97,208.34
9,MA,717,147807.29,206.15


## 5. Atraso na entrega por estado
% de pedidos entregues depois da data estimada (`order_delivered_customer_date > order_estimated_delivery_date`).

*Conceitos:* `CASE WHEN`, média de indicador 0/1.

In [10]:
q('''
SELECT
    c.customer_state                                             AS estado,
    COUNT(*)                                                     AS pedidos_entregues,
    SUM(CASE WHEN date(p.order_delivered_customer_date)
                  > date(p.order_estimated_delivery_date)
             THEN 1 ELSE 0 END)                                  AS pedidos_atrasados,  -- seu CASE WHEN
    ROUND(100.0 * SUM(CASE WHEN date(p.order_delivered_customer_date)
                               > date(p.order_estimated_delivery_date)
                          THEN 1 ELSE 0 END) / COUNT(*), 1)      AS pct_atraso          -- atrasados ÷ total
FROM pedidos p
JOIN clientes c ON p.customer_id = c.customer_id
WHERE p.order_status = 'delivered'
  AND p.order_delivered_customer_date IS NOT NULL                -- só quem tem data de entrega
GROUP BY c.customer_state
HAVING COUNT(*) >= 100                                           -- mesmo corte da pergunta 4
ORDER BY pct_atraso DESC
''')

,estado,pedidos_entregues,pedidos_atrasados,pct_atraso
0,AL,397,85,21.4
1,MA,717,125,17.4
2,SE,335,51,15.2
3,PI,476,66,13.9
4,CE,1279,176,13.8
5,BA,3256,396,12.2
6,RJ,12350,1495,12.1
7,PA,946,106,11.2
8,ES,1995,214,10.7
9,PB,517,54,10.4


## 6. Atraso derruba a nota?
Nota média de avaliação por faixa de atraso: no prazo, 1–3 dias, 4–7 dias, mais de 7 dias.

*Conceitos:* `julianday()` para diferença de datas, `CASE` para faixas.

In [11]:
q('''
-- ETAPA 1: dias de atraso de cada pedido entregue (sua subtração de datas)
WITH atraso AS (
    SELECT
        order_id,
        julianday(date(order_delivered_customer_date))
      - julianday(date(order_estimated_delivery_date))     AS dias_atraso   -- negativo = chegou antes
    FROM pedidos
    WHERE order_status = 'delivered'
      AND order_delivered_customer_date IS NOT NULL
),

-- ETAPA 2: nota de cada pedido (média, caso o pedido tenha mais de uma avaliação)
nota AS (
    SELECT order_id, AVG(review_score) AS nota
    FROM avaliacoes
    GROUP BY order_id
),

-- ETAPA 3: suas faixas com CASE WHEN + o join pela chave order_id
base AS (
    SELECT
        a.order_id,
        n.nota,
        CASE
            WHEN a.dias_atraso <= 0 THEN '1. No prazo'
            WHEN a.dias_atraso <= 3 THEN '2. 1 a 3 dias'
            WHEN a.dias_atraso <= 7 THEN '3. 4 a 7 dias'
            ELSE                         '4. Mais de 7 dias'
        END AS faixa_atraso
    FROM atraso a
    JOIN nota n ON a.order_id = n.order_id
)

-- ETAPA 4: nota média por faixa
SELECT
    faixa_atraso,
    COUNT(*)                                                  AS pedidos,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1)        AS pct_pedidos,
    ROUND(AVG(nota), 2)                                       AS nota_media,
    ROUND(100.0 * AVG(CASE WHEN nota <= 2 THEN 1 ELSE 0 END), 1) AS pct_notas_1_e_2
FROM base
GROUP BY faixa_atraso
ORDER BY faixa_atraso
''')

,faixa_atraso,pedidos,pct_pedidos,nota_media,pct_notas_1_e_2
0,1. No prazo,89443,93.3,4.29,9.2
1,2. 1 a 3 dias,1852,1.9,3.29,32.2
2,3. 4 a 7 dias,1748,1.8,2.11,67.6
3,4. Mais de 7 dias,2781,2.9,1.70,79.2


## 7. Taxa de recompra
Quantos clientes únicos (`customer_unique_id`) fizeram mais de um pedido?

*Conceitos:* subconsulta ou CTE com `COUNT(DISTINCT order_id)`.

In [12]:
q('''
-- ETAPA 1: quantos pedidos cada PESSOA fez
WITH pedidos_por_cliente AS (
    SELECT
        c.customer_unique_id,                         -- a pessoa (não o customer_id!)
        COUNT(DISTINCT p.order_id) AS qtd_pedidos
    FROM clientes c
    JOIN pedidos p ON c.customer_id = p.customer_id
    WHERE p.order_status = 'delivered'
    GROUP BY c.customer_unique_id
)

-- ETAPA 2: o percentual
SELECT
    COUNT(*)                                              AS clientes_total,
    SUM(CASE WHEN qtd_pedidos >= 2 THEN 1 ELSE 0 END)     AS clientes_recompra,
    ROUND(100.0 * SUM(CASE WHEN qtd_pedidos >= 2 THEN 1 ELSE 0 END)
          / COUNT(*), 2)                                  AS taxa_recompra_pct
FROM pedidos_por_cliente
''')

,clientes_total,clientes_recompra,taxa_recompra_pct
0,93358,2801,3.0


## 8. Coorte de primeira compra
Para cada mês de primeira compra, quantos clientes voltaram nos meses seguintes?

*Conceitos:* `MIN()` por cliente em uma CTE, JOIN de volta com pedidos.

In [13]:
coorte = q('''
-- ETAPA 1: cada pedido com a PESSOA e o mês da compra
WITH compras AS (
    SELECT
        c.customer_unique_id,
        date(p.order_purchase_timestamp) AS data_compra
    FROM clientes c
    JOIN pedidos p ON c.customer_id = p.customer_id
    WHERE p.order_status = 'delivered'
),

-- ETAPA 2: o seu MIN, a primeira compra de cada cliente = a coorte dele
primeira AS (
    SELECT customer_unique_id, MIN(data_compra) AS primeira_compra
    FROM compras
    GROUP BY customer_unique_id
),

-- ETAPA 3: o seu join de volta, quantos meses depois da 1ª compra cada pedido aconteceu
base AS (
    SELECT
        c.customer_unique_id,
        strftime('%Y-%m', f.primeira_compra) AS coorte,
        (CAST(strftime('%Y', c.data_compra) AS INT) - CAST(strftime('%Y', f.primeira_compra) AS INT)) * 12
      + (CAST(strftime('%m', c.data_compra) AS INT) - CAST(strftime('%m', f.primeira_compra) AS INT))
                                               AS meses_depois   -- 0 = mês da 1ª compra
    FROM compras c
    JOIN primeira f ON c.customer_unique_id = f.customer_unique_id
)

-- ETAPA 4: clientes distintos por coorte e por mês
SELECT coorte, meses_depois, COUNT(DISTINCT customer_unique_id) AS clientes
FROM base
GROUP BY coorte, meses_depois
ORDER BY coorte, meses_depois
''')
coorte.head(15)

,coorte,meses_depois,clientes
0,2016-09,0,1
1,2016-10,0,262
2,2016-10,6,1
3,2016-10,9,1
4,2016-10,11,1
5,2016-10,13,1
6,2016-10,15,1
7,2016-10,17,1
8,2016-10,19,2
9,2016-10,20,2


## 9. Formas de pagamento
Participação de cada `payment_type` no valor total e número médio de parcelas.

In [14]:
q('''
SELECT
    pg.payment_type                                           AS tipo_pagamento,
    COUNT(DISTINCT pg.order_id)                               AS pedidos,          -- sua ideia: qtd de pedidos
    ROUND(SUM(pg.payment_value), 2)                           AS valor_total,
    ROUND(100.0 * SUM(pg.payment_value)
          / SUM(SUM(pg.payment_value)) OVER (), 1)            AS participacao_pct, -- % do dinheiro total
    ROUND(AVG(pg.payment_installments), 1)                    AS parcelas_media    -- média de parcelas
FROM pagamentos pg
JOIN pedidos p ON pg.order_id = p.order_id
WHERE p.order_status = 'delivered'
GROUP BY pg.payment_type
ORDER BY valor_total DESC
''')

,tipo_pagamento,pedidos,valor_total,participacao_pct,parcelas_media
0,credit_card,74304,12101094.88,78.5,3.5
1,boleto,19191,2769932.58,18.0,1.0
2,voucher,3679,343013.19,2.2,1.0
3,debit_card,1485,208421.12,1.4,1.0


## 10. Tabela RFM por cliente *(vai alimentar a segmentação)*
Para cada `customer_unique_id`, considerando só pedidos entregues:
- **Recência:** dias entre a última compra do cliente e a última data da base
- **Frequência:** número de pedidos
- **Valor:** soma paga

Salve o resultado em uma tabela nova: `CREATE TABLE rfm AS ...`

In [15]:
# Apaga a tabela se ela já existir, para você poder rodar a célula de novo sem erro
con.execute("DROP TABLE IF EXISTS rfm")

con.execute('''
CREATE TABLE rfm AS

-- ETAPA 1: valor pago por pedido (soma as formas de pagamento do mesmo pedido)
WITH pago AS (
    SELECT order_id, SUM(payment_value) AS valor_pedido
    FROM pagamentos
    GROUP BY order_id
),

-- ETAPA 2: pedidos entregues, com a PESSOA e o valor
base AS (
    SELECT
        c.customer_unique_id,
        p.order_id,
        date(p.order_purchase_timestamp) AS data_compra,
        pg.valor_pedido
    FROM clientes c
    JOIN pedidos p ON c.customer_id = p.customer_id
    JOIN pago pg   ON p.order_id   = pg.order_id
    WHERE p.order_status = 'delivered'
),

-- ETAPA 3: a última data da base (a sua "data de referência")
referencia AS (
    SELECT MAX(data_compra) AS ultima_data_base FROM base
)

-- ETAPA 4: as três colunas por cliente único
SELECT
    b.customer_unique_id,
    CAST(julianday(r.ultima_data_base) - julianday(MAX(b.data_compra)) AS INT) AS recencia,   -- dias
    COUNT(DISTINCT b.order_id)                                                 AS frequencia, -- pedidos
    ROUND(SUM(b.valor_pedido), 2)                                              AS valor       -- R$
FROM base b
CROSS JOIN referencia r                   -- "cola" a data de referência em todas as linhas
GROUP BY b.customer_unique_id
''')
con.commit()   # grava de verdade no arquivo do Drive

# Conferência
q("SELECT * FROM rfm LIMIT 5")

,customer_unique_id,recencia,frequencia,valor
0,0000366f3b9a7992bf8c76cfdf3221e2,111,1,141.90
1,0000b849f77a49e4a4ce2b2a4ca5be3f,114,1,27.19
2,0000f46a3911fa3c0805444483337064,537,1,86.22
3,0000f6ccb0745a6a4b88665a16c9f078,321,1,43.62
4,0004aac84e0df4da2b147fca70cf8255,288,1,196.89


In [ ]:
# con.execute('''DROP TABLE IF EXISTS rfm''')
# con.execute('''CREATE TABLE rfm AS  ''')
# q('SELECT * FROM rfm LIMIT 5')

In [16]:
q('''
SELECT
    COUNT(*)                     AS clientes,
    MIN(recencia)  AS rec_min,   ROUND(AVG(recencia), 0)   AS rec_media,  MAX(recencia)  AS rec_max,
    MIN(frequencia) AS freq_min, ROUND(AVG(frequencia), 2) AS freq_media, MAX(frequencia) AS freq_max,
    MIN(valor)     AS val_min,   ROUND(AVG(valor), 2)      AS val_media,  MAX(valor)     AS val_max
FROM rfm
''')

,clientes,rec_min,rec_media,rec_max,freq_min,freq_media,freq_max,val_min,val_media,val_max
0,93357,0,237.0,695,1,1.03,15,9.59,165.2,13664.08
